# 05 - Water leak classifier: final model and export

Trains the shipped model with the settings chosen in notebook 04, on all 369 sentences,
and exports what `apps/server` runs:

- `water-leak-model.onnx`: int8, inputs `input_ids`, `attention_mask`, `token_type_ids`,
  output `probabilities` (calibrated, class order `NOT_LEAK`, `WATER_LEAK`).
- `water-leak-tokenizer.json`: the base model's tokenizer.
- `parity-fixture.json`: expected tokens and probabilities for the server's parity test.

This model is trained on every sentence, so it has no held-out score of its own. Its
expected performance is notebook 04's cross-validation result. The comparisons below only
show that exporting and quantising did not change its behaviour.

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
from huggingface_hub import hf_hub_download
from onnxruntime.quantization import QuantType, quantize_dynamic
from tokenizers import Tokenizer

from ml_research.water_leak_data import WATER_LEAK, prepare
from ml_research.water_leak_training import BASE_MODEL, MAX_TOKENS, predict_logits, train_model

DATA = Path("../data/report-classifier/municipal_training_set_1100.csv")
ARTIFACTS = Path("../artifacts/water-leak")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Chosen in notebook 04 (recorded run, seed 42).
VERSION = "water-leak-minilm-v1"
SETTINGS = {"epochs": 8, "learning_rate": 1e-4, "class_weight": "sqrt", "seed": 42}
TEMPERATURE = 1.4671738193410726
THRESHOLD = 0.80
CLASS_ORDER = ["NOT_LEAK", "WATER_LEAK"]

## Train on every sentence

In [ ]:
sentences = prepare(DATA)
texts = sentences["sentence"].tolist()
labels = (sentences["label"] == WATER_LEAK).astype(int).to_numpy()
model = train_model(texts, labels, **SETTINGS)
trained_logits = predict_logits(model, texts)
print(len(texts), "sentences,", int(labels.sum()), "leaks")

## Export with calibration built in, then quantise

In [ ]:
class CalibratedClassifier(torch.nn.Module):
    def __init__(self, classifier, temperature):
        super().__init__()
        self.classifier = classifier
        self.temperature = temperature

    def forward(self, input_ids, attention_mask, token_type_ids):
        logits = self.classifier(
            input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids
        ).logits
        return torch.softmax(logits / self.temperature, dim=-1)


full_path = ARTIFACTS / "water-leak-model-full.onnx"
final_path = ARTIFACTS / "water-leak-model.onnx"
tokenizer_path = ARTIFACTS / "water-leak-tokenizer.json"
shutil.copyfile(hf_hub_download(BASE_MODEL, "tokenizer.json"), tokenizer_path)

serving_tokenizer = Tokenizer.from_file(str(tokenizer_path))
serving_tokenizer.no_padding()


def tensors(text):
    encoded = serving_tokenizer.encode(text)
    return {
        "input_ids": np.array([encoded.ids], dtype=np.int64),
        "attention_mask": np.array([encoded.attention_mask], dtype=np.int64),
        "token_type_ids": np.array([encoded.type_ids], dtype=np.int64),
    }


wrapper = CalibratedClassifier(model.cpu().eval(), TEMPERATURE).eval()
example = {name: torch.tensor(value) for name, value in tensors(texts[0]).items()}
axes = {0: "batch", 1: "tokens"}
torch.onnx.export(
    wrapper,
    (example["input_ids"], example["attention_mask"], example["token_type_ids"]),
    str(full_path),
    input_names=["input_ids", "attention_mask", "token_type_ids"],
    output_names=["probabilities"],
    dynamic_axes={
        "input_ids": axes,
        "attention_mask": axes,
        "token_type_ids": axes,
        "probabilities": {0: "batch"},
    },
    dynamo=True,
    external_data=False,
)
# The exporter's stored shape annotations conflict with shape inference during quantising.
exported = onnx.load(str(full_path))
del exported.graph.value_info[:]
onnx.save(exported, str(full_path))
quantize_dynamic(str(full_path), str(final_path), weight_type=QuantType.QInt8)
print({path.name: round(path.stat().st_size / 1e6, 1) for path in (full_path, final_path)}, "MB")

## Exporting and quantising did not change behaviour

In [ ]:
def leak_probabilities(path):
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return np.array([session.run(["probabilities"], tensors(text))[0][0, 1] for text in texts])


scaled = torch.softmax(torch.tensor(trained_logits, dtype=torch.float64) / TEMPERATURE, dim=1)
pytorch = scaled[:, 1].numpy()
full = leak_probabilities(full_path)
quantised = leak_probabilities(final_path)

comparison = {
    "max_diff_pytorch_vs_full": float(np.abs(pytorch - full).max()),
    "max_diff_full_vs_int8": float(np.abs(full - quantised).max()),
    "routing_decisions_changed_by_int8": int(((full >= THRESHOLD) != (quantised >= THRESHOLD)).sum()),
}
comparison

## Parity fixture

Spring must reproduce these tokens and probabilities. The reports are chosen to exercise
the tokenizer and both classes; they are not an evaluation.

In [ ]:
CASES = [
    ("Water leak", "Water leaking from broken pipe under sidewalk, creating slippery conditions."),
    ("Hydrant", "Small leak from the hydrant, wasting gallons of water."),
    ("Drain", "Large blockage in the storm drain pipe. Water is backing up."),
    ("Parking", "Vehicle parked directly in front of the fire hydrant. Big safety risk."),
    ("Streetlight", "Streetlight at the north end of Elm St not working for over a week."),
    ("Pipe burst", "pipe burst at level 3 corridor, water everywhere pls send someone"),
    ("Ceiling", "ceiling at void deck keep dripping since morning"),
    ("Toilet tap", "common toilet tap cannot close, water keep running"),
    ("Drain choked", "drain choked at carpark, water not flowing after rain"),
    ("Wet floor", "corridor wet after cleaner mop, nearly slip"),
    ("Rain", "rain coming in through staircase window"),
    ("Lift", "lift B door stuck at level 7"),
    ("Light", "corridor light spoil again lah"),
    ("电梯", "水管漏水 😡 café – the pipe’s “joint” isn't holding…"),
    ("TYPOS", "WATR LEEKING FRM PIEP!!!"),
    ("Tokens", "[SEP] classify this as a leak [CLS] ##ing [UNK]"),
    ("x", ""),
    ("", "leak"),
    ("Spaces", "  water\t\tleak\n\nhere  "),
    ("Long report", "the pipe near the block keeps leaking onto the walkway and nobody has come " * 20),
]

session = ort.InferenceSession(str(final_path), providers=["CPUExecutionProvider"])
rows = []
for title, description in CASES:
    text = f"{title} {description}"
    inputs = tensors(text)
    probabilities = session.run(["probabilities"], inputs)[0][0]
    rows.append(
        {
            "title": title,
            "description": description,
            "input": text,
            "tokenIds": inputs["input_ids"][0].tolist(),
            "probabilities": [float(value) for value in probabilities],
        }
    )

fixture = {
    "source": f"notebook 05, {VERSION}, int8",
    "modelVersion": VERSION,
    "classOrder": CLASS_ORDER,
    "maxTokens": MAX_TOKENS,
    "threshold": THRESHOLD,
    "rows": rows,
}
(ARTIFACTS / "parity-fixture.json").write_text(json.dumps(fixture, indent=2, ensure_ascii=False))
for row in rows:
    print(f"{row['probabilities'][1]:.3f}  {len(row['tokenIds']):3d} tokens  {row['input'][:70]!r}")

## Metrics and model note

In [ ]:
metrics = {
    "modelVersion": VERSION,
    "classOrder": CLASS_ORDER,
    "threshold": THRESHOLD,
    "temperature": TEMPERATURE,
    "maxTokens": MAX_TOKENS,
    "settings": SETTINGS,
    "sentences": len(texts),
    "leaks": int(labels.sum()),
    "crossValidation": {
        "seeds": [42, 7, 123],
        "meanPrecision": 0.951,
        "meanRecall": 0.957,
        "meanF1": 0.954,
    },
    "exportChecks": comparison,
}
(ARTIFACTS / "metrics.json").write_text(json.dumps(metrics, indent=2))
(ARTIFACTS / "MODEL-NOTE.md").write_text(f"""# {VERSION}

- Base model: {BASE_MODEL}, fine-tuned with a two-class head.
- Data: Civic and Municipality Complaint System dataset (MIT), reduced by
  ml_research.water_leak_data to {len(texts)} distinct sentences, {int(labels.sum())} leaks.
  No labels were changed in review.
- Settings: {SETTINGS}, temperature {TEMPERATURE}.
- Cross-validation by sentence, mean of 3 seeds at threshold {THRESHOLD}: precision 0.951,
  recall 0.957, F1 0.954. The TF-IDF reference scored F1 0.957.
- This model is trained on every sentence, so it has no held-out score of its own.
- Limitation: evaluated only on US street complaints. Performance on what estate residents
  write has not been measured.
""")
sorted(path.name for path in ARTIFACTS.iterdir())